# **Real Estate Data Cleaning Documentation**


## **Section 1: Importing Dependencies**

In [1]:
import os
import googlemaps
import pandas as pd
from openai import OpenAI
from module_cleaning import *
from sklearn.impute import KNNImputer
from dotenv import load_dotenv

load_dotenv()

google_api_key = os.getenv("GOOGLE_MAPS_API_KEY")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

gmaps_client = googlemaps.Client(key=google_api_key)
OpRout_Client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=openrouter_api_key,)


## **Section 2: Initialization**
We load the raw data here and make sure the outputs don't truncated.

In [3]:
# Load the raw dataset
df_raw = pd.read_csv(r"C:\Users\Bal\OneDrive\Desktop\My Data Analysis\EDA - Real Estate\Data\raw_unclean_data.csv")
# Make sure to not trunctate results
pd.set_option('display.max_rows', None)          
pd.set_option('display.max_columns', None)        
pd.set_option('display.max_colwidth', None)      
pd.set_option('display.expand_frame_repr', False) 
pd.set_option('display.width', None)  

## **Section 3: Dealing with the Noise**
##### We remove duplicates and drop unnecessary features to reduce noise.

In [3]:
# Drop Duplicates
df = df_raw.drop_duplicates()
# Drop Unnecessary Features
df = df_raw.drop(columns = ["Description", "Latitude", "Longitude"])

## **Section 4: Parsing of Addresses**
##### We handle the null values by clustering locations based on City/Municipality and Province and filling them with their cluster median. But the Location feature still needs to be parsed into their Barangay, City, and Province. Since addresses are messy, we use Google Geocoding API (NLP-enabled) and an LLM (Open Router) to fill what Google misses. The function then returns the parsed components, which are added as new features to the dataset. 
##### *Other offline parsing methods also exist, such as libpostal, which is also accurate for Philippine addresses. However, it's RAM-intensive requiring 4GB for a small data set. So we will use this method as a temporary solution since it has zero costs during initial trials.*


In [4]:
# Google parse the addresses to their respective Barangays, City or Municipality, and Province
df_geocoded = geocode_addresses(df, gmaps_client, location_column="Location")
# Use LLM to further parse the addresses
df_parsed = LLM_treatment(df_geocoded, OpRout_Client)


LLM Processing: 100%|██████████| 398/398 [08:27<00:00,  1.28s/row]


## **Section 5: Treatment of Nulls**
##### Since the null percentage of the Location features is less than 1%, it's safe to just drop those rows. For Bedrooms and Bathrooms, we fill their nulls by the median of their location-based clusters (City/Municipality and Province), this avoids getting affected by the outliers. For the nulls Floor Area and Land Area, we use KNN imputation, which looks at similar houses and uses their average area instead of just taking their overall average.

#### **Null Count and Percentage**
##### This will be useful in our decision on how we treat the nulls of each features.

In [11]:
# Create Temporary DataFrame for Easy Readability 
null_info = pd.DataFrame({
    'Null Count': df_parsed.isnull().sum(),
    'Null Percentage %': df_parsed.isnull().sum() / len(df_parsed) * 100
})
# Display
null_info

,Null Count,Null Percentage %
Location,4,0.266667
Price,49,3.266667
Bedrooms,63,4.200000
Bathrooms,80,5.333333
Floor Area,37,2.466667
Land Area,23,1.533333
Town/Barangay,51,3.400000
Municipality/City,4,0.266667
Province/NCR,4,0.266667


In [12]:
# Create Copy
df_clean = df_parsed.copy()

# 1. With very low missing 0.27% on column Location, we drop it
df_clean = df_clean.dropna(subset=['Location']) 

# 2. Impute Price by Municipality/City and Province/NCR
df_clean['Price'] = df_clean.groupby(['Municipality/City', 'Province/NCR'])['Price'].transform(
    lambda x: x.fillna(x.median())
    
)


## **SECTION 6: Data Formatting and Simplification**
##### We standardize datatypes of features and make them easier to read. We then drop the unnecessary features for further analysis

In [13]:
# Convert data types safely
df_clean = df_clean.astype({
    'Bedrooms': 'float64',      # Change to 'float64' if you have half-baths (e.g., 2.5)
    'Bathrooms': 'float64',   # Floats handle half-baths perfectly
    'Floor Area': 'float64',  # Measurements will be floats
    'Land Area': 'float64',   # Measurements will be floats
    'Price': 'float64'        # Financial data will be float
})
# Make them easy to read by coverting to millions units
df_clean['Price'] = (df_clean['Price'] / 1_000_000).round(2)
df_clean.rename(columns={'Price': 'price_php_millions'}, inplace=True)

## **For Final Stage Cleaning**

In [ ]:
df_clean['Province/NCR'] = df_clean['Province/NCR'].str.replace('Lalawigan ng ', '', regex=False)
df_clean.loc[df_clean['Municipality/City'] == 'Makati', 'Municipality/City'] = 'Makati City'

## **Finalization:**
##### We review the changes occured to our dataset.

In [17]:
print(f'Raw data set shape: {df_raw.shape}', f'\nClean data set shape: {df_clean.shape}', f'\n')
print('Clean Dataset Null Counts:')
null_info = pd.DataFrame({
    'Null Count': df_clean.isnull().sum(),
    'Null Percentage %': df_clean.isnull().sum() / len(df_clean) * 100
})
display(null_info)
print('First 10 Rows of Our Dataset:')
df_clean.head(10)


Raw data set shape: (1500, 9) 
Clean data set shape: (1496, 9) 

Clean Dataset Null Counts:


,Null Count,Null Percentage %
Location,0,0.000000
price_php_millions,5,0.334225
Bedrooms,63,4.211230
Bathrooms,80,5.347594
Floor Area,37,2.473262
Land Area,23,1.537433
Town/Barangay,47,3.141711
Municipality/City,0,0.000000
Province/NCR,0,0.000000


First 10 Rows of Our Dataset:


,Location,price_php_millions,Bedrooms,Bathrooms,Floor Area,Land Area,Town/Barangay,Municipality/City,Province/NCR
0,"Santo Domingo, Cainta",9.50,4.0,3.0,144.00,136.0,Santo Domingo,Cainta,Rizal
1,"San Vicente, Santa Maria",4.40,3.0,2.0,63.20,80.0,San Vicente,Santa Maria,Bulacan
2,"Camella Tarlac Access Rd. Maliwalo, Tarlac",7.81,5.0,3.0,100.00,110.0,Maliwalo,Tarlac City,Tarlac
3,"Gabon, Abucay",NaN,3.0,2.0,67.00,88.0,Gabon,Abucay,Bataan
4,"Zambowood Rd. Boalan, Zamboanga",3.28,2.0,1.0,40.32,78.4,Boalan,Zamboanga City,Zamboanga del Sur
5,"Remulla Drive Tanza-Naic Rd. Sahud Ulan, Tanza",3.17,NaN,NaN,39.64,100.0,Sahud Ulan,Tanza,Cavite
6,"Santa Maria-San Jose Rd. Bulac, Santa Maria",1.80,2.0,2.0,52.00,40.0,Bulac,Santa Maria,Bulacan
7,"Aningway Sacatihan, Subic",8.43,5.0,3.0,142.00,121.0,Aningway Sacatihan,Subic,Zambales
8,"Zone 1 Brgy. Cadlan Pili Camarines Sur Cadlan, Pili",3.56,2.0,1.0,46.00,63.0,Cadlan,Pili,Camarines Sur
9,"Km. 19 West Service Rd. Cupang, Muntinlupa",28.65,3.0,3.0,219.30,90.0,Cupang,Muntinlupa,Metro Manila


##### Some towns or barangays remain unparsed (47 entries, or 3.14% of total) due to insufficient information. However, since these will only be used for secondary analysis and being only 3.14% of total, this will not compromise the overall data analysis.

In [16]:
df_clean.to_csv(r"C:\Users\Bal\OneDrive\Desktop\My Data Analysis\EDA - Real Estate\Data\Data_Cleaned.csv", index = False)